# Evaluación extendida: HD95, diagnóstico estadístico y figuras 3D

Continuación del notebook 13 con tres elementos adicionales:

1. **HD95** (percentil 95 de la distancia de Hausdorff, en mm reales y respetando el spacing anisotrópico de ImageCAS) para los tres modelos. Es la métrica de distancia de superficie habitual en el campo (ASOCA e ImageCAS evalúan con Dice y HD95).
2. **Diagnóstico de la discrepancia Wilcoxon / t-test** observada en la métrica de cobertura de la componente principal (baseline vs v1): histograma de diferencias pareadas, test de normalidad e identificación de casos atípicos que puedan condicionar el t-test.
3. **Figuras 3D** de los casos en los que v1 mejora más el clDice manteniendo un Dice similar al baseline, con comparación visual baseline vs v1 mediante la malla de marching cubes del notebook de visualización.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
import os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")
NNUNET_RAW = BASE / "nnUNet_raw"
NNUNET_RESULTS = BASE / "nnUNet_results"

os.environ["nnUNet_raw"] = str(NNUNET_RAW)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)


In [3]:
!pip install -q SimpleITK


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 14.6 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage, stats
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from skimage import measure
from tqdm.auto import tqdm

pd.set_option("display.max_columns", None)


## 1. Localización de las carpetas de validación de los tres modelos

Procedimiento idéntico al del notebook 13. Si la sesión conserva un `dfs` de una ejecución anterior, esta sección puede omitirse.


In [5]:
DATASET_NAME   = "Dataset501_ImageCAS"
PLANS_NAME     = "nnUNetPlans"
CONFIGURATION  = "3d_fullres"
FOLD           = 0

TRAINERS = {
    "Dice+CE (baseline)":        "nnUNetTrainer_100epochs",
    "Dice+CE+soft-clDice v1":    "nnUNetTrainerSoftClDice100epochs",
    "Dice+CE+soft-clDice v2":    "nnUNetTrainerSoftClDiceV2_100epochs",
}

def validation_folder(trainer_name):
    return (NNUNET_RESULTS / DATASET_NAME
            / f"{trainer_name}__{PLANS_NAME}__{CONFIGURATION}"
            / f"fold_{FOLD}" / "validation")

VAL_FOLDERS = {label: validation_folder(tr) for label, tr in TRAINERS.items()}
LABELS_DIR = NNUNET_RAW / DATASET_NAME / "labelsTr"

def list_case_ids(folder):
    return sorted(f.stem.replace(".nii", "") for f in folder.glob("*.nii.gz"))

common_cases = sorted(set(list_case_ids(list(VAL_FOLDERS.values())[0])))
print(len(common_cases), "casos de validacion")


40 casos de validacion


## 2. Función de HD95 (con spacing real en mm)

A diferencia de Dice y clDice, la distancia de Hausdorff depende del spacing físico de la imagen. Por ello, la imagen se carga con SimpleITK (y no solo como array de NumPy), lo que da acceso a `GetSpacing()` y permite calcular las distancias en milímetros reales en lugar de en número de vóxeles.


In [6]:
def get_surface_points(mask, spacing):
    mask = mask.astype(bool)
    eroded = ndimage.binary_erosion(mask)
    surface = mask & ~eroded
    coords_voxel = np.argwhere(surface)
    coords_mm = coords_voxel * np.array(spacing)
    return coords_mm


def hausdorff_95(pred, gt, spacing):
    """HD95 simetrica en mm: max(percentil95(pred->gt), percentil95(gt->pred)).
    Convencion estandar (ASOCA, ImageCAS)."""
    pred_pts = get_surface_points(pred, spacing)
    gt_pts = get_surface_points(gt, spacing)

    if len(pred_pts) == 0 or len(gt_pts) == 0:
        return np.nan

    tree_gt = cKDTree(gt_pts)
    tree_pred = cKDTree(pred_pts)

    dists_pred_to_gt, _ = tree_gt.query(pred_pts)
    dists_gt_to_pred, _ = tree_pred.query(gt_pts)

    return max(np.percentile(dists_pred_to_gt, 95), np.percentile(dists_gt_to_pred, 95))


def hausdorff_95_largest_component(pred, gt, spacing, connectivity=3):
    """Version alternativa: HD95 calculada solo sobre la componente conexa
    mas grande de la prediccion, ignorando fragmentos/falsos positivos
    aislados. Se calcula ademas de la version cruda porque, como se
    demostro con un caso de control sintetico, un unico fragmento
    fantasal de apenas ~1600 voxeles desconectado del arbol principal
    puede disparar el HD95 crudo a mas de 50mm, aunque el resto de la
    prediccion sea perfecta -- una manifestacion mas del problema de
    fragmentacion ya documentado con la metrica de cobertura."""
    structure = ndimage.generate_binary_structure(pred.ndim, connectivity)
    labeled, n = ndimage.label(pred.astype(bool), structure=structure)
    if n == 0:
        return np.nan
    sizes = ndimage.sum(pred, labeled, range(1, n + 1))
    largest_label = np.argmax(sizes) + 1
    pred_largest = (labeled == largest_label).astype(np.uint8)
    return hausdorff_95(pred_largest, gt, spacing)


def filter_small_fragments(mask, min_size=20, connectivity=3):
    """Filtra SOLO fragmentos diminutos (ruido de pocos voxeles),
    conservando cualquier componente de tamano razonable -- incluidas
    ramas reales desconectadas del tronco principal. A diferencia de
    hausdorff_95_largest_component, esta version NO descarta cobertura
    real: un experimento de control confirmo que quedarse solo con la
    componente mas grande tira ramas reales enteras y EMPEORA el HD95
    (29mm en el caso de control), mientras que filtrar solo el ruido
    diminuto mantiene el HD95 correcto (~0mm) cuando la fragmentacion
    es de ramas reales, no de falsos positivos aislados."""
    structure = ndimage.generate_binary_structure(mask.ndim, connectivity)
    labeled, n = ndimage.label(mask.astype(bool), structure=structure)
    if n == 0:
        return mask
    sizes = ndimage.sum(mask, labeled, range(1, n + 1))
    keep_labels = [i + 1 for i, s in enumerate(sizes) if s >= min_size]
    return np.isin(labeled, keep_labels).astype(np.uint8)


def hausdorff_95_filtered(pred, gt, spacing, min_size=20):
    pred_filtered = filter_small_fragments(pred, min_size=min_size)
    return hausdorff_95(pred_filtered, gt, spacing)


def load_mask_and_spacing(path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)
    spacing_xyz = img.GetSpacing()          # SimpleITK: (x, y, z)
    spacing_zyx = spacing_xyz[::-1]         # el array de numpy esta en orden (z, y, x)
    return (arr > 0).astype(np.uint8), spacing_zyx


def dice_score(pred, gt, eps=1e-6):
    pred = pred.astype(bool); gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    return (2.0*inter+eps)/(pred.sum()+gt.sum()+eps)


## 3. Cálculo de HD95 para los tres modelos (40 casos cada uno)

El cálculo es más lento que el de Dice y clDice, ya que construye KDTrees sobre los puntos de superficie de cada caso (varios minutos en total).


In [7]:
hd95_results = []

for label, folder in VAL_FOLDERS.items():
    for case_id in tqdm(common_cases, desc=f"HD95: {label}"):
        pred, spacing = load_mask_and_spacing(folder / f"{case_id}.nii.gz")
        gt, _ = load_mask_and_spacing(LABELS_DIR / f"{case_id}.nii.gz")

        hd95 = hausdorff_95(pred, gt, spacing)
        hd95_fixed = hausdorff_95_largest_component(pred, gt, spacing)
        hd95_filt = hausdorff_95_filtered(pred, gt, spacing, min_size=20)
        dice = dice_score(pred, gt)

        hd95_results.append({
            "model": label, "case_id": case_id,
            "dice": dice, "hd95_mm": hd95,
            "hd95_largest_component_mm": hd95_fixed,
            "hd95_filtered_mm": hd95_filt,
        })

df_hd95 = pd.DataFrame(hd95_results)
df_hd95.head()


HD95: Dice+CE (baseline):   0%|          | 0/40 [00:00<?, ?it/s]

HD95: Dice+CE+soft-clDice v1:   0%|          | 0/40 [00:00<?, ?it/s]

HD95: Dice+CE+soft-clDice v2:   0%|          | 0/40 [00:00<?, ?it/s]

,model,case_id,dice,hd95_mm,hd95_largest_component_mm,hd95_filtered_mm
0,Dice+CE (baseline),ImageCAS_0011,0.697345,57.271229,58.367558,57.275144
1,Dice+CE (baseline),ImageCAS_0017,0.711042,41.710288,55.426869,41.533561
2,Dice+CE (baseline),ImageCAS_0018,0.709800,39.059224,68.956896,39.060057
3,Dice+CE (baseline),ImageCAS_0021,0.609545,68.745402,72.694219,68.748316
4,Dice+CE (baseline),ImageCAS_0028,0.785743,6.426929,74.114984,7.280596


In [8]:
summary_hd95 = df_hd95.groupby("model").agg(
    dice_mean=("dice", "mean"),
    hd95_mean_mm=("hd95_mm", "mean"),
    hd95_median_mm=("hd95_mm", "median"),
    hd95_std_mm=("hd95_mm", "std"),
    hd95_lc_mean_mm=("hd95_largest_component_mm", "mean"),
    hd95_lc_median_mm=("hd95_largest_component_mm", "median"),
    hd95_filt_mean_mm=("hd95_filtered_mm", "mean"),
    hd95_filt_median_mm=("hd95_filtered_mm", "median"),
).round(4)
summary_hd95


,dice_mean,hd95_mean_mm,hd95_median_mm,hd95_std_mm,hd95_lc_mean_mm,hd95_lc_median_mm,hd95_filt_mean_mm,hd95_filt_median_mm
model,,,,,,,,
Dice+CE (baseline),0.7459,35.8110,38.7899,21.2618,62.2135,62.3344,35.8716,38.7686
Dice+CE+soft-clDice v1,0.7549,33.7428,38.0662,21.4208,63.8228,62.7808,33.7829,38.0662
Dice+CE+soft-clDice v2,0.7482,33.9775,34.7821,19.0590,63.4132,62.0327,33.9614,34.7828


In [9]:
# Estadística pareada — HD95

import pandas as pd
from scipy.stats import wilcoxon

# Una fila por paciente, una columna por modelo
hd95_wide = df_hd95.pivot(
    index="case_id",
    columns="model",
    values="hd95_mm"
)

print("Pacientes:", len(hd95_wide))
print("Modelos:", hd95_wide.columns.tolist())

comparisons = [
    ("Dice+CE (baseline)", "Dice+CE+soft-clDice v1"),
    ("Dice+CE (baseline)", "Dice+CE+soft-clDice v2"),
    ("Dice+CE+soft-clDice v1", "Dice+CE+soft-clDice v2"),
]

results = []

for model_a, model_b in comparisons:

    a = hd95_wide[model_a]
    b = hd95_wide[model_b]

    valid = a.notna() & b.notna()
    a = a[valid]
    b = b[valid]

    # B - A
    # Diferencia negativa: B presenta menor HD95 (mejor)
    diff = b - a

    stat, p = wilcoxon(a, b)

    results.append({
        "comparacion": f"{model_a} → {model_b}",
        "n": len(a),
        "HD95_A_mean_mm": a.mean(),
        "HD95_B_mean_mm": b.mean(),
        "delta_B_minus_A_mm": diff.mean(),
        "median_delta_mm": diff.median(),
        "casos_mejoran": int((diff < 0).sum()),
        "casos_empeoran": int((diff > 0).sum()),
        "casos_iguales": int((diff == 0).sum()),
        "wilcoxon_stat": stat,
        "p_value": p,
        "significativo_p<0.05": p < 0.05
    })

results_hd95 = pd.DataFrame(results)

display(results_hd95.round(4))

Pacientes: 40
Modelos: ['Dice+CE (baseline)', 'Dice+CE+soft-clDice v1', 'Dice+CE+soft-clDice v2']


,comparacion,n,HD95_A_mean_mm,HD95_B_mean_mm,delta_B_minus_A_mm,median_delta_mm,casos_mejoran,casos_empeoran,casos_iguales,wilcoxon_stat,p_value,significativo_p<0.05
0,Dice+CE (baseline) → Dice+CE+soft-clDice v1,40,35.8110,33.7428,-2.0683,-1.2548,32,8,0,132.0,0.0001,True
1,Dice+CE (baseline) → Dice+CE+soft-clDice v2,40,35.8110,33.9775,-1.8336,-0.1783,21,19,0,359.0,0.5014,False
2,Dice+CE+soft-clDice v1 → Dice+CE+soft-clDice v2,40,33.7428,33.9775,0.2347,0.3378,18,22,0,325.0,0.2591,False
